# Day 3 Session 1 -- RAG Exercises

In the demos you saw how embeddings encode semantic meaning into vectors, how ChromaDB indexes those vectors for fast retrieval, and how section-aware chunking preserves document structure. Now you will combine these pieces into a reusable `SearchEngine` class that your consulting knowledge platform can call at query time.

In [1]:
!pip install -q openai chromadb python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


---
## Setup

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

import openai
import chromadb
import numpy as np

print("All imports successful!")

All imports successful!


---
## Recap

Embeddings turn text into dense vectors where cosine similarity reflects semantic closeness. ChromaDB stores those vectors in an HNSW index so that `collection.query()` returns the nearest neighbours in sub-millisecond time. Section-aware chunking splits on `## ` headers first, keeping each logical section intact for higher-quality retrieval.

---
## Task 1: Build a SearchEngine Class (Embeddings + ChromaDB)

Build a `SearchEngine` class that an engineer can drop into any service. It should:
1. Accept a list of consulting knowledge documents in `__init__`
2. Embed all documents using OpenAI embeddings and index them in a ChromaDB collection
3. Provide a `search(query, k)` method that uses `collection.query()` to return the top-k results with their distances

**Requirements:**
- Use `chromadb.Client()` and `get_or_create_collection` with `cosine` distance
- Embed documents via `openai.OpenAI().embeddings.create()`
- The `search` method must embed the query and call `collection.query()`
- Return a list of `(distance, document)` tuples sorted by distance (lowest first)

In [3]:
# Task 1 -- Build a SearchEngine class backed by ChromaDB

class SearchEngine:
    def __init__(self, documents):
        """Embed all documents and index them in a ChromaDB collection."""
        self.client = openai.OpenAI()
        self.chroma = chromadb.Client()
        self.collection = self.chroma.get_or_create_collection(
            name="search_engine_kb",
            metadata={"hnsw:space": "cosine"}
        )

        # TODO: Embed all documents using self.client.embeddings.create()
        #       model = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
        # TODO: Add documents + embeddings + ids to self.collection via .add()
        #       ids can be [f"doc_{i}" for i in range(len(documents))]
        pass

    def search(self, query, k=3):
        """Return the top-k most similar documents as (distance, text) tuples."""
        # TODO: Embed the query using self.client.embeddings.create()
        # TODO: Call self.collection.query(query_embeddings=[query_emb], n_results=k)
        # TODO: Zip results['distances'][0] with results['documents'][0]
        # TODO: Return list of (distance, document) tuples (already sorted by ChromaDB)
        pass


# --------------- Test ---------------
corpus = [
    "Digital transformation in financial services requires a phased approach starting with core banking modernization.",
    "Post-merger integration success depends on Day-1 readiness and a structured 100-day plan for synergy capture.",
    "Omnichannel retail strategy must unify inventory, pricing, and customer data across physical and digital channels.",
    "Private equity value creation levers include revenue growth, operational efficiency, and strategic bolt-on acquisitions.",
    "Supply chain resilience requires diversification of sourcing, near-shoring critical components, and end-to-end visibility.",
    "Organizational health, as measured by the OHI, is the strongest predictor of sustained long-term performance.",
    "ESG strategy must be embedded into core business operations and capital allocation to create shareholder value.",
    "Cross-border M&A transactions require careful due diligence on regulatory, tax, and cultural integration risks."
]

# engine = SearchEngine(corpus)
# for query in [
#     "What are best practices for post-merger integration?",
#     "How should a retailer approach omnichannel transformation?",
#     "What are the key risks in cross-border M&A?"
# ]:
#     print(f"\nQuery: {query}")
#     results = engine.search(query, k=3)
#     for dist, doc in results:
#         print(f"  {dist:.4f} | {doc}")

---
## Expected Output

When you uncomment the test block above and run it, you should see output similar to:

```
Indexed 8 documents in search_engine_kb

Query: What are best practices for post-merger integration?
  0.3842 | Post-merger integration success depends on Day-1 readiness and a structured 100-day plan for synergy capture.
  0.5765 | Cross-border M&A transactions require careful due diligence on regulatory, tax, and cultural integration risks.
  0.6700 | Private equity value creation levers include revenue growth, operational efficiency, and strategic bolt-on acquisitions.

Query: How should a retailer approach omnichannel transformation?
  0.3535 | Omnichannel retail strategy must unify inventory, pricing, and customer data across physical and digital channels.
  0.4778 | Digital transformation in financial services requires a phased approach starting with core banking modernization.
  0.5739 | Organizational health, as measured by the OHI, is the strongest predictor of sustained long-term performance.

Query: What are the key risks in cross-border M&A?
  0.2713 | Cross-border M&A transactions require careful due diligence on regulatory, tax, and cultural integration risks.
  0.6283 | Post-merger integration success depends on Day-1 readiness and a structured 100-day plan for synergy capture.
  0.6883 | Supply chain resilience requires diversification of sourcing, near-shoring critical components, and end-to-end visibility.
```

*Exact distances will vary slightly between runs but the ranking order should be stable. Lower distance = more similar (cosine distance, not cosine similarity).*